In [1]:
import open_clip

In [2]:
import numpy as np
import pandas as pd

In [3]:
# pretrained also accepts local paths
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k') 

In [4]:
model.eval()
#token 数
context_length = model.context_length
#clip可以识别的唯一单词或符号的数量。
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Context length:", context_length)
print("Vocab size:", vocab_size)

Model parameters: 151,277,313
Context length: 77
Vocab size: 49408


In [5]:
preprocess

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x00000207235F0180>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)

In [6]:
from open_clip import tokenizer

In [7]:
import os
import skimage
import IPython.display
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from collections import OrderedDict
import torch

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


In [8]:
import zipfile
import os

# 设置 zip 文件路径
zip_path = 'ImageNet/ImageNet-Sketch/ImageNet-Sketch.zip'  # 注意：确保扩展名是 .zip

# 设置解压目标路径
extract_path = 'ImageNet/ImageNet-Sketch'

# 解压文件
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("解压完成！")


解压完成！


In [9]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ImageNetSketchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(root, file))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image


In [10]:
dataset = ImageNetSketchDataset(
    root_dir='ImageNet/ImageNet-Sketch/sketch',
    transform=preprocess
)

In [12]:
import os

# 设置解压后的图像根目录
image_root = 'ImageNet/ImageNet-Sketch/sketch'  # 根据你的实际解压路径调整

# 支持的图像扩展名
image_extensions = ('.png', '.jpg', '.jpeg', '.bmp')

# 统计图像数量
image_count = 0
for root, _, files in os.walk(image_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_count += 1

print(f"解压后的图像总数：{image_count} 张")


📷 解压后的图像总数：50889 张


In [13]:
# 文件名: extract_logits_from_sketch.py

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()


In [14]:

# ============ 包装 Dataset，确保图像转为 Tensor ============
class ImageNetSketchDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.to_tensor = transforms.ToTensor()

In [ ]:



    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img)
        if isinstance(img, Image.Image):
            img = self.to_tensor(img)
        return img, label


    def __len__(self):
        return len(self.base_dataset)


    @property
    def targets(self):
        return self.base_dataset.targets

In [ ]:


# 包装原始 Dataset
cifar100_train_dataset = TensorizedDataset(cifar100_train_dataset)
cifar100_test_dataset = TensorizedDataset(cifar100_test_dataset)


# ============ 文本特征 ============
print("提取文本特征中...")
text_descriptions = [f"A photo of a {label}" for label in cifar100.classes]
text_tokens = tokenizer.tokenize(text_descriptions).to(device)


with torch.no_grad():
    text_features = model.encode_text(text_tokens).float()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
text_features_np = text_features.cpu().numpy()


# # ============ 获取每个 dataset 的 1/100 子集 ============
# small_train_dataset = Subset(cifar100_train_dataset, list(range(len(cifar100_train_dataset) // 100)))
# small_test_dataset = Subset(cifar100_test_dataset, list(range(len(cifar100_test_dataset) // 100)))


# ============ 提取图像特征 ============
import torch.nn.functional as F
def extract_features(dataset, name=""):
    features = []
    for img_tensor, _ in tqdm(dataset, desc=f"[{name}] 提取图像特征"):
        resized = F.interpolate(img_tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
        img_input = resized.to(device)
        with torch.no_grad():
            image_feature = model.encode_image(img_input).float()
            image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)
        features.append(image_feature.squeeze(0).cpu())
    return torch.stack(features).numpy()


train_features = extract_features(cifar100_train_dataset, "训练集")
test_features = extract_features(cifar100_test_dataset, "测试集")


# ============ 标签 ============
train_labels = np.array([cifar100_train_dataset[i][1] for i in range(len(cifar100_train_dataset))]).reshape(-1, 1)
test_labels = np.array([cifar100_test_dataset[i][1] for i in range(len(cifar100_test_dataset))]).reshape(-1, 1)


# ============ 计算 logits ============
print("计算 logits...")
temperature = 100.0
train_logits = temperature * train_features @ text_features_np.T
test_logits = temperature * test_features @ text_features_np.T


# ============ 保存为指定格式 ============
output_data = ((train_logits, train_labels), (test_logits, test_labels))
output_dir = "logits_debug"
os.makedirs(output_dir, exist_ok=True)
save_path = join(output_dir, "probs_vit_b32_c100_logits_debug.p")


with open(save_path, "wb") as f:
    pickle.dump(output_data, f)


print(f"✅ zong样本测试完成，结果保存在：{save_path}")